# DLICV - full reproduction

This notebook reproduces the entire project - all configs, all 5 variants (A/B/C/D/E) across 4 models x 3 seeds, the full N_synth sweep, and all plots and analyses (aggregation, CLIP/DINOv2 filter, N_synth curve, Grad-CAM).

## How to run on Colab

1. **Runtime -> Change runtime type -> T4 GPU**.
2. Set `REPRO_LEVEL` in the first cell, then **Runtime -> Run all**.
3. **L3 only:** **Settings -> Secrets -> add `HF_TOKEN`** (read access). L2 works without a token - `micwuj/dlicv-synth` is public.

## Reproduction levels (`REPRO_LEVEL` switch in section 0)

| Level | What it does | HF_TOKEN |
|---|---|---|
| **`L2`** (default) | Pulls ready-made synthetics (D + E + LoRA) from HF, **trains everything from scratch**, makes all plots | optional |
| **`L3`** (full) | Generates variant D from scratch (SD 1.5), **trains LoRA** for 5 classes, generates variant E, filters (CLIP+DINOv2), then trains everything | required |

## What gets produced (`outputs/_aggregate/`)

- `master_table.csv`, `summary_by_model_variant.csv`, `per_class_accuracy.csv`, `top_confused_pairs.csv`
- `test_accuracy_bars.png`, `training_curves.png`, `confusion_matrices_mean.png`, `per_class_delta_heatmap.png`
- `n_synth_sweep/` - test_acc(N_synth) curve for 4 models
- `filters/` - scatter/histograms/retention CLIP vs DINOv2 (D and E)
- `gradcam/` - resnet18 + convnext_tiny x {Ragdoll, staffordshire} x A/B/D/E

## 0. Configuration and setup

In [ ]:
# REPRODUCTION LEVEL
# 'L2' = pull ready-made synthetics from HF + train everything
# 'L3' = generate variant D + train LoRA + variant E + filter + train
REPRO_LEVEL = 'L2'

REPO_URL = 'https://github.com/micwuj/DLICV.git'
REPO_DIR = '/content/dlicv'
HF_REPO_ID = 'micwuj/dlicv-synth'

# Parameters - used only for L3
BREEDS_LORA = ['Ragdoll', 'Birman', 'Maine_Coon', 'staffordshire_bull_terrier', 'american_bulldog']
BREEDS_NO_LORA = ['Siamese', 'Persian', 'boxer', 'english_cocker_spaniel', 'english_setter']
N_PER_CLASS_D = 300
N_PER_CLASS_E = 300
N_IMAGES_PER_BREED = 10   # real images per class for LoRA training
LORA_STEPS = 500
LORA_RANK = 16
LORA_LR = 1e-4

assert REPRO_LEVEL in ('L2', 'L3'), REPRO_LEVEL
print('REPRO_LEVEL =', REPRO_LEVEL)

In [ ]:
import os
if not os.path.exists(REPO_DIR):
    !git clone {REPO_URL} {REPO_DIR}
%cd {REPO_DIR}
!git pull

In [ ]:
# Full requirements + peft/safetensors (needed for training)
!pip install -q -r requirements.txt peft==0.13.2 safetensors==0.4.5

In [ ]:
from pathlib import Path
PETS = Path('data/raw/oxford-iiit-pet/images')
if not (PETS.exists() and any(PETS.iterdir())):
    !python scripts/download_pets.py
else:
    print('Pets present:', PETS)

In [ ]:
from huggingface_hub import login, whoami
HF_TOKEN = None
try:
    from google.colab import userdata
    HF_TOKEN = userdata.get('HF_TOKEN')
except Exception:
    HF_TOKEN = os.environ.get('HF_TOKEN')

# micwuj/dlicv-synth is public - L2 works without a token (anonymous pull).
# L3 requires a token to download the base SD 1.5 model.
if HF_TOKEN:
    login(token=HF_TOKEN, add_to_git_credential=False)
    print('HF logged in as:', whoami()['name'])
elif REPRO_LEVEL == 'L3':
    raise RuntimeError('L3 requires HF_TOKEN (base SD model) - add it in Colab Secrets (key HF_TOKEN, read access)')
else:
    print('no HF_TOKEN - micwuj/dlicv-synth is public, pulling anonymously')

In [ ]:
from src.utils.device import get_device, device_info
d = get_device()
print('device:', device_info(d))
if d.type != 'cuda':
    print('WARNING: no GPU - training and generation will be very slow. Runtime -> Change runtime type -> T4 GPU.')
!pytest tests/test_smoke.py tests/test_transforms.py tests/test_models.py tests/test_metrics.py -q

## 1. Synthetic data

- **L2:** we pull ready-made `variant_D` (simple prompts), `variant_E` (LoRA mix) + manifests + filter scores from HF.
- **L3:** we generate everything from scratch (cells 1b-1c below). The L2 cell is skipped.


In [ ]:
# L2: pull ready-made synthetics from HF
import zipfile, shutil
from huggingface_hub import hf_hub_download

def pull_variant(variant):
    root = Path(f'data/synthetic/variant_{variant}')
    sample_png = root / 'Ragdoll' / 'Ragdoll_0000.png'
    if not sample_png.exists():
        zip_path = hf_hub_download(repo_id=HF_REPO_ID, repo_type='dataset',
                                   filename=f'variant_{variant}.zip', local_dir='data/synthetic')
        root.mkdir(parents=True, exist_ok=True)
        with zipfile.ZipFile(zip_path) as zf:
            zf.extractall(root)
        nested = root / f'variant_{variant}'
        if nested.exists() and nested.is_dir():
            for dd in nested.iterdir():
                tgt = root / dd.name
                if tgt.exists():
                    shutil.rmtree(tgt)
                shutil.move(str(dd), str(tgt))
            nested.rmdir()
    # fetch manifests separately - they were empty in older zips
    hf_hub_download(repo_id=HF_REPO_ID, repo_type='dataset',
                    filename=f'variant_{variant}/manifest_kept.csv', local_dir='data/synthetic')
    n_pngs = sum(1 for _ in root.rglob('*.png'))
    with open(root / 'manifest_kept.csv') as f:
        n_rows = sum(1 for _ in f) - 1
    print(f'variant_{variant}: {n_pngs} pngs, manifest {n_rows} rows')
    assert n_rows > 2000 and sample_png.exists()

if REPRO_LEVEL == 'L2':
    pull_variant('D')
    pull_variant('E')
    for f in ['variant_D/filter_scores.csv', 'variant_E/filter_scores_lora.csv']:
        try:
            hf_hub_download(repo_id=HF_REPO_ID, repo_type='dataset', filename=f, local_dir='data/synthetic')
        except Exception as e:
            print('warn: failed to download', f, '-', e)
else:
    print('L3: skipping pull, generating from scratch in cells 1b-1c')

### 1b. (L3 only) Generate variant D + filter

Simple SD 1.5 prompts for all 10 classes, then a CLIP+DINOv2 filter (floor 0.65 / 0.55) that writes `manifest_kept.csv` + `filter_scores.csv`. For L2 these cells are skipped.


In [ ]:
# L3: filter helper (CLIP + DINOv2 nearest-sim to real images, floors) -> manifest_kept.csv + scores
import csv, json

def filter_variant(variant, breeds_to_score, recycle_breeds=None, scores_name='filter_scores.csv'):
    from src.synth.filter import FilterConfig, compute_embeddings, nearest_sim
    root = Path(f'data/synthetic/variant_{variant}')
    split = json.load(open('data/splits/pets_10cls_30perclass_seed0.json'))
    class_to_idx = split['meta']['class_to_idx']
    pets = Path('data/raw/oxford-iiit-pet/images')
    FLOOR_CLIP, FLOOR_DINOV2 = 0.65, 0.55
    fc = FilterConfig()
    dev = get_device()
    kept, scores = [], []
    for breed in breeds_to_score:
        label = class_to_idx[breed]
        real = [pets / e['image'] for e in split['train'] if e['label'] == label]
        synth = sorted((root / breed).glob(f'{breed}_*.png'))
        rc = compute_embeddings(real, 'clip', fc.clip_model_id, dev, 16)
        rd = compute_embeddings(real, 'dinov2', fc.dinov2_model_id, dev, 16)
        sc = compute_embeddings(synth, 'clip', fc.clip_model_id, dev, 16)
        sd = compute_embeddings(synth, 'dinov2', fc.dinov2_model_id, dev, 16)
        cs = nearest_sim(sc, rc, fc.top_k).numpy()
        ds = nearest_sim(sd, rd, fc.top_k).numpy()
        mask = (cs >= FLOOR_CLIP) & (ds >= FLOOR_DINOV2)
        print(f'  {breed}: kept {int(mask.sum())}/{len(synth)}')
        for p, ci, di, k in zip(synth, cs, ds, mask):
            scores.append({'breed': breed, 'path': p.as_posix(), 'clip_sim': float(ci),
                           'dinov2_sim': float(di), 'kept': bool(k)})
            if k:
                kept.append({'breed': breed, 'path': p.as_posix()})
    if recycle_breeds:   # recycle variant D manifest rows for classes without LoRA
        for r in csv.DictReader(open('data/synthetic/variant_D/manifest_kept.csv')):
            if r['breed'] in recycle_breeds:
                kept.append({'breed': r['breed'],
                             'path': r['path'].replace('variant_D/', f'variant_{variant}/')})
    with open(root / 'manifest_kept.csv', 'w', newline='') as f:
        w = csv.DictWriter(f, fieldnames=['breed', 'path']); w.writeheader(); w.writerows(kept)
    with open(root / scores_name, 'w', newline='') as f:
        w = csv.DictWriter(f, fieldnames=['breed', 'path', 'clip_sim', 'dinov2_sim', 'kept'])
        w.writeheader(); w.writerows(scores)
    print(f'variant_{variant}: total kept {len(kept)} -> manifest_kept.csv')

print('helper filter_variant ready')

In [ ]:
# L3: generate variant D (all 10 classes) + filter
if REPRO_LEVEL == 'L3':
    get_ipython().system(f'python scripts/generate_synthetic.py --variant D --n-per-class {N_PER_CLASS_D}')
    from src.synth.prompts import BREEDS
    filter_variant('D', list(BREEDS.keys()))
else:
    print('L2: skip')

### 1c. (L3 only) LoRA + variant E + filter

Train SD 1.5 LoRA for the 5 hardest classes (from 10 real images/class), generate variant E with LoRA for those classes, reuse variant D for the remaining 5 classes, then CLIP+DINOv2 filter -> `manifest_kept.csv` + `filter_scores_lora.csv`.


In [ ]:
# L3: data preparation + LoRA training (saved to data/loras/<breed>.safetensors)
if REPRO_LEVEL == 'L3':
    breeds_arg = ' '.join(BREEDS_LORA)
    get_ipython().system(f'python scripts/prepare_lora_data.py --breeds {breeds_arg} --n-per-breed {N_IMAGES_PER_BREED}')
    for breed in BREEDS_LORA:
        out = Path('data/loras') / f'{breed}.safetensors'
        if out.exists():
            print(f'{breed}: LoRA already exists, skip')
            continue
        print(f'training LoRA: {breed}')
        get_ipython().system(
            f'python scripts/train_lora.py --breed {breed} '
            f'--train-data-dir data/lora_train/{breed} --output-dir data/loras '
            f'--steps {LORA_STEPS} --rank {LORA_RANK} --lr {LORA_LR}')
else:
    print('L2: skip')

In [ ]:
# L3: generate variant E (LoRA for 5 classes) + reuse D for 5 classes without LoRA + filter
if REPRO_LEVEL == 'L3':
    lora_arg = ' '.join(BREEDS_LORA)
    get_ipython().system(
        f'python scripts/generate_synthetic.py --variant E --n-per-class {N_PER_CLASS_E} '
        f'--lora-dir data/loras --breeds {lora_arg}')
    ve, vd = Path('data/synthetic/variant_E'), Path('data/synthetic/variant_D')
    ve.mkdir(parents=True, exist_ok=True)
    for b in BREEDS_NO_LORA:
        dst = ve / b
        if dst.exists() and any(dst.glob('*.png')):
            continue
        dst.mkdir(parents=True, exist_ok=True)
        for f in (vd / b).glob(f'{b}_*.png'):
            shutil.copy(f, dst / f.name)
        print(f'{b}: copied from variant_D')
    filter_variant('E', BREEDS_LORA, recycle_breeds=BREEDS_NO_LORA, scores_name='filter_scores_lora.csv')
else:
    print('L2: skip')

## 2. Generate all configs

In [ ]:
# Splits, 60 standard configs + N_synth sweep (4 models x {30,60,240} x 3 seeds)
!python scripts/make_all_splits.py
!python scripts/generate_configs.py
!python scripts/generate_sweep_configs.py
n_cfg = len(list(Path('configs/exp').glob('*.yaml')))
print(f'configs total: {n_cfg}')

## 3. Train everything

Per model (fastest to slowest), `--skip-existing`. The pattern `*_<model>*.yaml` catches the standard A/B/C/D/E plus the N_synth sweep for that model. 96 runs total.

In [ ]:
# dinov2_small
!python -u scripts/run_all.py --pattern '*_dinov2_small*.yaml' --overrides configs/colab.yaml --skip-existing

In [ ]:
# resnet18
!python -u scripts/run_all.py --pattern '*_resnet18*.yaml' --overrides configs/colab.yaml --skip-existing

In [ ]:
# deit_tiny
!python -u scripts/run_all.py --pattern '*_deit_tiny*.yaml' --overrides configs/colab.yaml --skip-existing

In [ ]:
# convnext_tiny
!python -u scripts/run_all.py --pattern '*_convnext_tiny*.yaml' --overrides configs/colab.yaml --skip-existing

## 4. Aggregation + all plots

In [ ]:
!python scripts/aggregate_results.py

In [ ]:
# Filter (CLIP vs DINOv2 vs joint) - no retraining
!PYTHONPATH=. python scripts/analyze_filters.py

In [ ]:
# N_synth vs accuracy curve (4 models)
!PYTHONPATH=. python scripts/plot_n_synth_sweep.py

In [ ]:
# Grad-CAM: resnet18 + convnext_tiny x {Ragdoll, staffordshire} x A/B/D/E
!PYTHONPATH=. python scripts/run_gradcam.py

In [ ]:
import pandas as pd
print('=== Summary by (model, variant) ===')
print(pd.read_csv('outputs/_aggregate/summary_by_model_variant.csv').to_string())

In [ ]:
import matplotlib.pyplot as plt
import matplotlib.image as mpimg

pngs = sorted(Path('outputs/_aggregate').rglob('*.png'))
print(f'{len(pngs)} plots:')
for p in pngs:
    img = mpimg.imread(p)
    h, w = img.shape[0], img.shape[1]
    plt.figure(figsize=(13, 13 * h / w))
    plt.imshow(img); plt.axis('off')
    plt.title(str(p.relative_to('outputs/_aggregate')), fontsize=10)
    plt.show()